# 🎮 Pokemon Jump & Run

### Controls
- **Arrow Keys / WASD** — Move
- **Space / Up / W** — Jump
- **F** — Attack (Donnerschock / Schallwelle)
- **Stomp enemies** from above!

### 🕹️ Play the Game

In [ ]:
# Play the game
game_html = open("./builtin/jump_and_run.html").read()
displayHTML(game_html)

### 💾 Save Stats
Copy the JSON from the game output, paste it below, and click Save.

In [ ]:
import json
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display as ipy_display
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, FloatType, TimestampType
)

schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("duration_seconds", FloatType(), True),
    StructField("score", IntegerType(), True),
    StructField("won", BooleanType(), True),
    StructField("lives_left", IntegerType(), True),
    StructField("deaths_total", IntegerType(), True),
    StructField("deaths_enemy", IntegerType(), True),
    StructField("deaths_water", IntegerType(), True),
    StructField("deaths_fall", IntegerType(), True),
    StructField("coins_collected", IntegerType(), True),
    StructField("enemies_stomped", IntegerType(), True),
    StructField("enemies_zapped", IntegerType(), True),
    StructField("attacks_used", IntegerType(), True),
    StructField("jumps", IntegerType(), True),
    StructField("forms_collected", StringType(), True),
    StructField("final_form", StringType(), True),
    StructField("max_x_reached", IntegerType(), True),
])

textarea = widgets.Textarea(
    placeholder='Paste the stats JSON here (Ctrl+V)...',
    layout=widgets.Layout(width='100%', height='80px')
)
btn = widgets.Button(description='💾 Save Stats', button_style='primary')
output = widgets.Output()

def on_save(b):
    output.clear_output()
    with output:
        txt = textarea.value.strip()
        if not txt or not txt.startswith("{"):
            print("⚠️ Paste valid JSON first.")
            return
        raw = json.loads(txt)
        raw["timestamp"] = datetime.fromisoformat(raw["timestamp"].replace("Z", "+00:00"))
        raw["duration_seconds"] = float(raw["duration_seconds"])
        row = [tuple(raw[f.name] for f in schema.fields)]
        df = spark.createDataFrame(row, schema=schema)
        if not spark.catalog.tableExists("game_stats"):
            df.write.format("delta").saveAsTable("game_stats")
        else:
            df.write.mode("append").format("delta").saveAsTable("game_stats")
        print(f"✅ Saved! Score: {raw['score']} | Won: {raw['won']} | Duration: {raw['duration_seconds']}s")
        textarea.value = ""

btn.on_click(on_save)
ipy_display(widgets.VBox([widgets.Label('📋 Paste game stats JSON:'), textarea, btn, output]))

### 📋 All Game Stats & Summary

In [ ]:
if spark.catalog.tableExists("game_stats"):
    display(spark.sql("SELECT * FROM game_stats ORDER BY timestamp DESC"))

    spark.sql("""
        SELECT
            COUNT(*) AS games_played,
            SUM(CASE WHEN won THEN 1 ELSE 0 END) AS wins,
            ROUND(SUM(CASE WHEN won THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS win_rate_pct,
            MAX(score) AS high_score,
            ROUND(AVG(score), 0) AS avg_score,
            ROUND(SUM(duration_seconds), 1) AS total_playtime_sec,
            ROUND(AVG(duration_seconds), 1) AS avg_duration_sec,
            SUM(deaths_total) AS total_deaths,
            SUM(enemies_stomped) AS total_stomps,
            SUM(enemies_zapped) AS total_zaps,
            SUM(coins_collected) AS total_coins,
            SUM(jumps) AS total_jumps
        FROM game_stats
    """).show(truncate=False)
else:
    print("⚠️ No game_stats table yet.")